# Emergency Vision AI: Production Person-Crop Training, Evaluation & GPU Benchmark

This notebook provides the unified, reproducible Google Colab GPU workflow for **Emergency Vision AI**.

### Architecture & Single Source of Truth
- **GitHub**: Sole source of truth for all code, scripts, configurations, and canonical notebook.
- **Google Drive**: Persistent authoritative dataset storage (`/content/drive/MyDrive/emergency-vision-ai/data/urfd`) and backup destination for trained models and benchmark results.
- **Google Colab**: Ephemeral GPU execution runtime (Tesla T4) that bootstraps itself idempotently.
- **Second-Stage Action Training**: Aligns training with production inference by extracting 16-frame person-crop tubes (`YOLO11n` + `ByteTrack` + 5% padding) with strict sequence-level isolation (Seed=42).
- **Workflow**: **Bootstrap → Train → Evaluate → Benchmark → Summary**.

## 1. Bootstrap Environment & Google Drive

Clones or synchronizes the repository from GitHub, mounts Google Drive, verifies the authoritative dataset (`30 FALL + 40 NORMAL` videos), materializes Git LFS models, installs production dependencies, creates the local dataset symlink, and generates a complete environment status report.

In [ ]:
# ==============================================================================
# 1. BOOTSTRAP ENVIRONMENT & GOOGLE DRIVE
# ==============================================================================
import os
REPO_DIR = "/content/emergency-vision-ai"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mukhammadiev01-1/emergency-vision-ai.git {REPO_DIR}

%cd {REPO_DIR}

import scripts.colab_bootstrap as bootstrap
bootstrap.run_bootstrap()


## 2. Second-Stage Person-Crop Action Training

Trains/fine-tunes R3D-18 on production-style 16-frame person crops generated via YOLO11n + ByteTrack with hard-negative mining and strict sequence isolation (Seed=42: 49 train, 10 val, 11 test).

Saves best model to `models/action_recognition/r3d18_urfd_person_crops.pth` and automatically creates a persistent backup in Google Drive.

In [ ]:
# ==============================================================================
# 2. SECOND-STAGE PERSON-CROP ACTION TRAINING (ONE COMMAND)
# ==============================================================================
!python scripts/train_person_crop_pipeline.py \
    --dataset-root data/urfd \
    --base-checkpoint models/action_recognition/r3d18_urfd_best.pth \
    --yolo-model models/detection/yolo11n.pt \
    --output-dir models/action_recognition \
    --checkpoint-name r3d18_urfd_person_crops.pth \
    --epochs 12 \
    --batch-size 8 \
    --lr 1e-4 \
    --device cuda \
    --seed 42 \
    --drive-backup-dir /content/drive/MyDrive/emergency-vision-ai/models/action_recognition


## 3. Production Pipeline Comparative Evaluation

Evaluates both checkpoints side-by-side on the **actual production pipeline** (YOLO11n → ByteTrack → Person Crop → R3D-18 → Temporal Confirmation) across `fall-01..05` and `adl-01..05`.

Generates a side-by-side comparison of max $P(\text{FALL})$, confirmed emergency events, recall, and false positive rates.

In [ ]:
# ==============================================================================
# 3. PRODUCTION PIPELINE COMPARATIVE EVALUATION
# ==============================================================================
!python scripts/evaluate_production_pipeline.py \
    --action-model models/action_recognition/r3d18_urfd_person_crops.pth \
    --compare-with models/action_recognition/r3d18_urfd_best.pth \
    --dataset-root data/urfd \
    --max-fall-videos 5 \
    --max-normal-videos 5 \
    --device cuda \
    --output-json results/eval/pipeline_comparison.json


## 4. Production Pipeline GPU Benchmark (Tesla T4)

Measures end-to-end detection latency, person crop preprocessing, R3D-18 inference latency, and overall pipeline throughput (FPS) on the GPU accelerator.

In [ ]:
# ==============================================================================
# 4. PRODUCTION PIPELINE GPU BENCHMARK (Tesla T4)
# ==============================================================================
!python scripts/benchmark_gpu.py \
    --video data/urfd/videos/fall/fall-01-cam0.mp4 \
    --action-model models/action_recognition/r3d18_urfd_person_crops.pth \
    --yolo-model models/detection/yolo11n.pt \
    --device cuda \
    --threshold 0.70 \
    --interval 8 \
    --warmup 5 \
    --output-json results/benchmark_gpu_results.json


## 5. Final Artifacts & Google Drive Summary

Verifies the saved model checkpoints and structured JSON reports in both local workspace and Google Drive.

In [ ]:
# ==============================================================================
# 5. FINAL ARTIFACTS & GOOGLE DRIVE SUMMARY
# ==============================================================================
import os
import shutil
import glob

print("=" * 80)
print("      SAVING & VERIFYING EXPERIMENT ARTIFACTS IN GOOGLE DRIVE")
print("=" * 80)

drive_root = "/content/drive/MyDrive/emergency-vision-ai"

# Canonical artifacts mapping: (local_path, drive_relative_path)
artifacts_to_persist = [
    (
        "models/action_recognition/r3d18_urfd_person_crops.pth",
        "models/action_recognition/r3d18_urfd_person_crops.pth",
    ),
    (
        "models/action_recognition/r3d18_urfd_person_crops_metadata.json",
        "models/action_recognition/r3d18_urfd_person_crops_metadata.json",
    ),
    (
        "results/training/train_person_crops_results.json",
        "results/training/train_person_crops_results.json",
    ),
    (
        "results/eval/pipeline_comparison.json",
        "results/eval/pipeline_comparison.json",
    ),
    (
        "results/benchmark_gpu_results.json",
        "results/benchmark_gpu_results.json",
    ),
]

if os.path.exists(drive_root):
    print(f"Authoritative Google Drive Root: {drive_root}\n")
    for local_src, drive_rel in artifacts_to_persist:
        drive_dest = os.path.join(drive_root, drive_rel)
        if os.path.exists(local_src):
            os.makedirs(os.path.dirname(drive_dest), exist_ok=True)
            if not os.path.exists(drive_dest) or os.path.getsize(local_src) != os.path.getsize(drive_dest):
                shutil.copy2(local_src, drive_dest)
                print(f"  ✓ Copied to Drive: {drive_rel}")
            else:
                print(f"  ✓ Verified in Drive: {drive_rel}")
        else:
            print(f"  ⚠ Local artifact missing: {local_src}")
else:
    print(f"⚠ Google Drive not mounted at {drive_root}. Artifacts remain in local Colab runtime.")

print("\n" + "=" * 80)
print("EXPERIMENT READY FOR LOCAL ANTIGRAVITY SYNC:")
print("Run the following command on your local Mac:")
print("  python3 scripts/sync_experiment_results.py")
print("=" * 80)
